# Setka Cup ML Training Notebook\n\nUse this notebook in Google Colab to train the Setka prediction models with scikit-learn or XGBoost.\n\nModels trained here: winner classifier, first-set Over 18.5 classifier, total-points regressor, first-set-points regressor.

## 1. Clone your GitHub repo\n\nAfter pushing the project to GitHub, replace the URL below with your repository URL.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/setka-prediction-app.git'  # change this\nPROJECT_DIR = 'setka-prediction-app'\n\n!rm -rf $PROJECT_DIR\n!git clone $REPO_URL\n%cd $PROJECT_DIR

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt

## 3. Confirm data files\n\nIf the CSV files are not committed to the repo, upload them into the `data/` folder using the Colab file browser.

In [ ]:
from pathlib import Path\n\ndata_dir = Path('data')\nprint('Data files:')\nfor path in data_dir.glob('*.csv'):\n    print(' -', path, f'{path.stat().st_size/1_000_000:.1f} MB')

## 4. Train models\n\nUse `algorithm='auto'` to use XGBoost when available. For faster tests, set `max_training_rows=50000`. For full training, set it to `None`.

In [ ]:
from src.setka_core import load_raw_data\nfrom src.ml_pipeline import train_model_bundle, metrics_table, save_model_bundle\n\nmatches, leaderboard = load_raw_data('data')\nbundle = train_model_bundle(\n    matches,\n    algorithm='auto',\n    max_training_rows=120_000,  # change to None for all rows\n)\nmetrics_table(bundle)

## 5. Test a matchup prediction

In [ ]:
from src.ml_pipeline import predict_with_bundle\n\nplayer_a = 'Serhii Ponomarenko'\nplayer_b = 'Maksym Badai'\npredict_with_bundle(bundle, player_a, player_b, first_set_line=18.5, total_points_line=75.5)

## 6. Save model artifact\n\nYou can download this `.joblib` file and place it in `models/` for the Streamlit app to load.

In [ ]:
output_path = save_model_bundle(bundle, 'models/setka_ml_bundle.joblib')\nprint(output_path)\n\nfrom google.colab import files\nfiles.download(str(output_path))